# 04 · 全模态上传端口 + 真实 STEP 几何驱动报价

每种数据都有上传端口 (`/v1/upload/{email,step,audio,pdf,excel,image,auto}`)，统一 `/v1/rfq/intake` 一次收全模态并跑黄金链。STEP 走真实 OCP B-rep 几何 → 重量/表面积驱动报价。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from fastapi.testclient import TestClient
from services.api_server import app
from services import file_intake as fi
c = TestClient(app)
print('health:', c.get('/health').json())

## 各模态上传端口

In [ ]:
S = ROOT/'data'/'samples'
eml = c.post('/v1/upload/email', files={'file': open(S/'sample_rfq.eml','rb')}).json()
print('EMAIL  subject=', eml.get('subject'), '| src=', eml.get('_source'))
st = c.post('/v1/upload/step', files={'file': open(S/'sample_part.STEP','rb')}, data={'material':'6061'}).json()
g = st['geometry']
print('STEP   bbox=', g.get('bbox_mm'), 'vol_mm3=', round(g.get('volume_mm3') or 0,1), 'weight_kg=', g.get('weight_kg'), 'src=', g.get('volume_source'))
print('C1     holes=', st.get('features',{}).get('hole_count'), 'partial=', st.get('features',{}).get('partial'))

## 统一 intake: email + 真STEP → 几何驱动报价 (对比纯文本报价)

In [ ]:
# 纯文本 (默认 weight 0.5kg)
r_text = c.post('/v1/rfq/intake', data={'email_text':'50 pcs 6061 aluminum, 100x50x10mm, anodizing, IT7.','customer_name':'Northwind'}).json()['result']
# email + 真实 STEP (OCP 体积 → 真实重量)
r_step = c.post('/v1/rfq/intake',
    files={'email_file': open(S/'sample_rfq.eml','rb'), 'step_file': open(S/'sample_part.STEP','rb')},
    data={'customer_name':'Northwind','material_hint':'6061'}).json()['result']
print('纯文本  unit=', r_text['quote'].get('unit_price'), 'final=', r_text['quote'].get('final_price'))
print('STEP几何 unit=', r_step['quote'].get('unit_price'), 'final=', r_step['quote'].get('final_price'))
print('state=', r_step['state'], 'engine=', r_step['engine_source'], 'audit_ok=', r_step['audit_valid'])

## HITL 授权推进 (契约端点)

In [ ]:
cid = c.post('/v1/rfq/intake', data={'email_text':'10 pcs TC4 titanium fixtures, 60x30x12mm, as-machined, precision tolerance IT5.','customer_name':'Precision Med'}).json()['context_id']
print('before approve:', c.get(f'/v1/rfq/{cid}').json()['state'])
ap = c.post(f'/v1/rfq/{cid}/approve', data={'approver':'chief_engineer','comment':'grinding OK'}).json()
print('after  approve:', ap['state'], '| approved=', ap['approved'])
assert ap['state'] == 'DONE'
print('HITL→approve→DONE 断言通过 ✅')